[Guide Index](README.md)


# 06.03 OWL Manchester Syntax as RDF

`starlayer.starontology.manchester` can represent the structure of an OWL 2 Manchester Syntax document as an RDF graph, using the `manch:` ontology (specific to StarLayer). The graph is a human-readable syntax tree of the document, with its frames, clauses, class and property expressions and data ranges, designed for storage, validation, viewing and editing. It can be built from text, edited as RDF, validated against the `manch:` SHACL shapes, and rendered back to text.

See the [Manchester Syntax guide](02.08-manchester-syntax.ipynb) for the notation itself and for `parse()` and `serialize()` with `format="manchester"`. This guide is about treating the *document itself* as an RDF syntax tree. The [SPARQL queries, updates and rules as RDF guide](06.02-sparql-and-srl-as-rdf.ipynb) does the same for queries, updates and rule sets.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). No extra runtime dependencies are needed. Run cells from top to bottom; later cells reuse variables from earlier ones.


In [ ]:
from starlayer.graph import Namespace, OWL, RDFS
from starlayer.graph import StarLayerGraph

EX = Namespace("http://example.org/vehicles#")


## Index of functions

OWL Manchester Syntax as RDF in `starlayer.starontology.manchester` has four functions.


| Function | What it does | Discussed in |
|---|---|---|
| `manchester_to_tree()` | Parses Manchester Syntax text into a `manch:` syntax tree graph. | section 06.03.2 |
| `tree_to_manchester()` | Renders a syntax tree graph back as Manchester Syntax text. | section 06.03.4 |
| `tree_to_owl()` | Compiles a syntax tree graph to the OWL RDF its text means. | section 06.03.5 |
| `validate_manchester_tree()` | Validates a tree against the `manch:` shapes, with RDFS reasoning, and returns a validation report. | section 06.03.6 |


## 06.03.1 A Manchester Syntax document's own structure, as RDF

A Manchester Syntax document has a structure of its own: frames, clauses, class and property expressions, and data ranges. `starlayer.starontology.manchester` represents that structure as an RDF graph, the document's syntax tree, using the `manch:` ontology. The tree can be inspected, edited and validated as data, and rendered back to text. This is separate from the notation in the [Manchester Syntax guide](02.08-manchester-syntax.ipynb), which parses text straight to OWL RDF and back. It follows the same pattern the [SPARQL queries, updates and rules as RDF guide](06.02-sparql-and-srl-as-rdf.ipynb) uses for SPARQL queries and SRL rule sets. The `manch:` vocabulary and shapes are standalone `.ttl` files (`starlayer/starontology/manchester/manchester-ast-ontology.ttl` and `manchester_shapes.ttl`), usable by any RDF tool.

The tree matches the parser's own internal structure. Spellings that mean the same thing, such as `onlysome`, `that` and `SuperClassOf:`, are written as the same tree as their long forms. The exception is `SuperClassOf:`/`SuperPropertyOf:`, which carry a `manch:reversed` flag, because which keyword was written changes the compiled OWL, not just how it reads.

Ordered children (frames, clauses, entries, operands) carry their position as a `tree:ord` ordinal, and a name or a literal is held in a `tree:Item`. There is no `rdf:List`.

The tree functions work on a `StarLayerGraph`. A graph holds exactly one document, so there is no separate root to pass.

Below, one small ontology fragment is parsed to OWL RDF. The following sections use it as the starting point, so that the effect of each edit shows up as a real, different compiled meaning and not just a change in triple count.

In [14]:
HEADER = 'Prefix: : <http://example.org/vehicles#>\n'

text = HEADER + """
Class: Vehicle
Class: Car
    SubClassOf: Vehicle

ObjectProperty: hasWheelCount
"""

# the fragment compiled to OWL RDF, the way the Manchester Syntax guide does it
original = StarLayerGraph()
original.bind("", EX)
original.bind("owl", OWL)
original.bind("rdfs", RDFS)
original.parse(data=text, format="manchester")
print(original.serialize(format="turtle", pretty=True))

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Vehicle a owl:Class .

:hasWheelCount a owl:ObjectProperty .



## 06.03.2 Parse straight to `manch:` RDF

`manchester_to_tree(text, tree_graph=None)` parses Manchester Syntax text into its `manch:` syntax tree. It uses the real parser's tokenizer, frame, clause and expression grammar unchanged, and captures the syntax tree as RDF instead of discarding it after compiling to OWL. It returns the graph. Pass `tree_graph=StarLayerGraph()` to build the tree in that graph; without it a new `StarLayerGraph` is built. Text that is not Manchester Syntax cannot be parsed and raises `FormatError` (kind `"manchester"`; see section 02.08.4 of the [Manchester Syntax guide](02.08-manchester-syntax.ipynb)). A document that parses can still be wrong; `validate_manchester_tree()` reports that (see section 06.03.6).

Below, the document from section 06.03.1 is parsed into a tree and the tree is printed.

In [15]:
from starlayer.graph import StarLayerGraph
from starlayer.starontology.manchester import manchester_to_tree

graph = manchester_to_tree(text, tree_graph=StarLayerGraph())
print('encoded triples:', len(graph))
print(graph.serialize(format='turtle', pretty=True, only_used=True))


encoded triples: 25
@prefix manch: <https://github.com/hidden-graph/starlayer/ns/graph/manchester-ast#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

_:b0 a manch:OntologyDocument ;
    manch:frames [
        a manch:ClassFrame ;
        manch:subject <http://example.org/vehicles#Vehicle> ;
        tree:ord 0
    ], [
        a manch:ClassFrame ;
        manch:clauses [
            a manch:ClassSubClassOfClause ;
            manch:items [
                a manch:ClauseItem ;
                manch:value <http://example.org/vehicles#Vehicle> ;
                tree:ord 0
            ] ;
            tree:ord 0
        ] ;
        manch:subject <http://example.org/vehicles#Car> ;
        tree:ord 1
    ], [
        a manch:ObjectPropertyFrame ;
        manch:subject <http://example.org/vehicles#hasWheelCount> ;
        tree:ord 2
    ] ;
    manch:prefixes [
        a manch:PrefixDeclaration ;
        manch:namespace "http://example.org/vehicles#" ;
        manch:p

A graph holds one document. Below, parsing a second document into the graph that already holds one raises a `ValueError`; a new document goes into a fresh graph.

In [ ]:
try:
    manchester_to_tree(text, tree_graph=graph)
except ValueError as error:
    print("ValueError:", error)

ValueError: the graph already holds a manch:OntologyDocument: a graph holds one Manchester syntax tree, parse into a fresh graph


## 06.03.3 Edit the document by editing the RDF

A tree is an ordinary RDF graph, so it can be edited with standard graph operations. Edits to the graph structure change the document the tree represents. No text is searched or replaced.

Below, a new item is appended to the `manch:items` of `Car`'s `SubClassOf:` clause with `starlayer.graph.ordinals.append`, which gives it the next ordinal. The new item, `hasWheelCount some xsd:integer`, becomes a second superclass expression.

In [16]:
from starlayer.graph import BlankNode, RDF, XSD, ordinals
from starlayer.starontology.manchester import MANCH

sub_class_of_clause = next(graph.subjects(RDF.type, MANCH.ClassSubClassOfClause))

# manch:Some's own onProperty/filler shape (see manchester-ast-ontology.ttl)
some_expr = BlankNode()
graph.add((some_expr, RDF.type, MANCH.Some))
graph.add((some_expr, MANCH.onProperty, EX.hasWheelCount))
graph.add((some_expr, MANCH.filler, XSD.integer))

new_item = BlankNode()
graph.add((new_item, RDF.type, MANCH.ClauseItem))
graph.add((new_item, MANCH.value, some_expr))

# append new_item to the clause's manch:items: it gets the next tree:ord
ordinals.append(graph, sub_class_of_clause, MANCH.items, new_item)
print('items after edit:', len(ordinals.ordered_items(graph, sub_class_of_clause, MANCH.items)))

items after edit: 2


## 06.03.4 Render back to Manchester Syntax text

`tree_to_manchester(tree_graph)` renders a tree back to Manchester Syntax text. It is the inverse of the parser's own structure, so the edited tree can be rendered with no intermediate Python object to edit instead. The edit in section 06.03.3 is made on the RDF graph only.

Below, the edited tree is rendered. `Car`'s `SubClassOf:` clause shows both items on one line. IRIs are written in full: the tree stores IRIs, not the prefixed names the original text used.

In [17]:
from starlayer.starontology.manchester import tree_to_manchester

edited_text = tree_to_manchester(graph)
print(edited_text)


Prefix: : <http://example.org/vehicles#>

Class: <http://example.org/vehicles#Vehicle>

Class: <http://example.org/vehicles#Car>
    SubClassOf: <http://example.org/vehicles#Vehicle>, <http://example.org/vehicles#hasWheelCount> some <http://www.w3.org/2001/XMLSchema#integer>

ObjectProperty: <http://example.org/vehicles#hasWheelCount>



## 06.03.5 Prove the edit was structural

`tree_to_owl(tree_graph)` compiles a tree to the OWL RDF its text means, returned as a `StarLayerGraph`. It renders the tree to text and compiles that text the same way `parse(format="manchester")` does.

Below, the edited tree is compiled and compared with the original fragment from section 06.03.1. The two should differ: the compiled graph gains the new `owl:someValuesFrom` restriction on `Car`. That shows the edit changed the meaning of the ontology, not just its layout.

In [18]:
from starlayer.graph.compare import isomorphic
from starlayer.starontology.manchester import tree_to_owl

edited_owl = tree_to_owl(graph)

print('original triple count:', len(original))
print('edited triple count:  ', len(edited_owl))
print('isomorphic (should be False - the edit is a real semantic change):', isomorphic(original, edited_owl))

original triple count: 4
edited triple count:   8
isomorphic (should be False - the edit is a real semantic change): False


## 06.03.6 Validate the edited document, with RDFS reasoning over the ontology

`validate_manchester_tree(tree_graph)` validates a tree against the `manch:` shapes and returns a validation report. It checks the asserted graph first, for problems that would stop the later steps, such as a node that lost its asserted type. It then runs the shapes with `inference="rdfs"` and the `manch:` ontology, so real RDFS reasoning happens before SHACL validation. That is what lets a dispatch shape like `manch:ClassExpressionShape` check a single abstract superclass instead of listing every concrete tag. `sqe_validate()` and `srl_validate()` use the same approach for their trees.

Below, the edited tree is validated and conforms. Then one triple is deliberately corrupted: the spliced-in `manch:Some` restriction is retyped as something that is not a recognized class-expression tag. The report fails with a real SHACL violation, not a silent pass. The report text is long because it shows the shape and the focus node in full.

In [19]:
from starlayer.starontology import manchester

report = manchester.validate_manchester_tree(graph)
print('conforms:', report.conforms)

conforms: True


In [20]:
graph.remove((some_expr, RDF.type, MANCH.Some))

report = manchester.validate_manchester_tree(graph)
print('conforms after corruption (should be False):', report.conforms)
print('report graph is an', type(report.report_graph).__name__)
print(report.report_text)

conforms after corruption (should be False): False
report graph is an StarLayerGraph
Validation Report
Conforms: False

Pre-flight problems (asserted graph, checked before inference):
  - node has manch:filler, onProperty but no asserted concrete manch: type (node b339f2310856b2de0578b5c560c9452a)

Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:NodeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:node manch:ClassItemShape ; sh:path manch:items ]
	Focus Node: [ manch:items [ manch:value <http://example.org/vehicles#Vehicle> ; rdf:type manch:ClauseItem, rdfs:Resource ; tree:ord 0 ], [ manch:value [ manch:filler xsd:integer ; manch:onProperty <http://example.org/vehicles#hasWheelCount> ; rdf:type rdfs:Resource ] ; rdf:type manch:ClauseItem, rdfs:Resource ; tree:ord 1 ] ; rdf:type manch:ClassSubClassOfClause, manch:Clause, rdfs:Resource ; tree:ord 0 ]
	Value Node: [ manch:value [ manch:filler xsd:integer ; manch:onProperty <http://example.org/vehi